# IOAI — 2025 Team Selection Day3 Nlp Code Difficulty (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day3-nlp-code-difficulty'
for f in ['train.csv','val.csv','test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
# 의존성: lightgbm (Jupyter/Colab 환경에 없으면 설치)
try:
    import lightgbm  # noqa: F401
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm  # noqa: F401

# Day 3 — Code Difficulty · 모범답안 (원본 대회 코드)

> **Kazakhstan – Team Selection 2025 · Day 3 (ML on text)**

이 노트북은 Batyr Yerdenov 의 **실제 TST 제출 코드**(github.com/batyrq/ioai-tst-kazakhstan)를 거의 **그대로** 옮긴 것입니다.
데이터는 공개 up-solving 미러(`tst-day-3-upsolving`)의 원본을 동봉본으로 사용하며, 바뀐 부분은 **데이터 경로**와 **로컬 채점용 검증 예측 추가**뿐입니다(주석으로 표시). 더 정돈·향상된 재구현은 **모범답안2** 탭을 참고하세요.


In [ ]:
# --- 데이터 경로 어댑터(추가): 동봉 데이터 폴더로 이동 (원본 코드는 그대로) ---
# 현재 폴더(".")를 먼저 확인 → 모범답안 작업본(_solution)에 머물러 제출물이 거기에 저장되게 한다
# (상위 폴더에도 데이터가 풀려 있어 "."보다 ".."를 먼저 보면 self(내 풀이) 폴더에 잘못 저장됨).
import os
for _b in [".", "..", "../.."]:
    if os.path.exists(os.path.join(_b, "train.csv")):
        os.chdir(_b); break
print("data dir:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score

def extract_code_features(df):
    df['char_len'] = df['code'].str.len()
    df['word_count'] = df['code'].apply(lambda x: len(x.split()))
    df['sentence_count'] = df['code'].apply(lambda x: x.count('.') + x.count('\n'))
    df['avg_token_len'] = df['code'].apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    df['num_digits'] = df['code'].apply(lambda x: sum(c.isdigit() for c in x))
    df['num_loops'] = df['code'].str.count(r'\bfor\b|\bwhile\b')
    df['num_if'] = df['code'].str.count(r'\bif\b')
    df['num_functions'] = df['code'].str.count(r'\bdef\b|\bfunction\b|\bvoid\b')
    df['num_comments'] = df['code'].str.count(r'#|//|/\*|\*/')
    df['has_O_complexity'] = df['code'].str.contains(r'O\([^)]+\)', flags=re.IGNORECASE).astype(int)
    df['contains_tree'] = df['code'].str.contains(r'\btree\b', flags=re.IGNORECASE).astype(int)
    df['contains_dp'] = df['code'].str.contains(r'dynamic programming|\bdp\b', flags=re.IGNORECASE).astype(int)
    df['contains_hash'] = df['code'].str.contains(r'\bhash', flags=re.IGNORECASE).astype(int)
    df['contains_stack'] = df['code'].str.contains(r'\bstack\b', flags=re.IGNORECASE).astype(int)
    df['contains_recursive'] = df['code'].str.contains(r'\brecursive|\brecursion', flags=re.IGNORECASE).astype(int)
    df['has_algo_steps'] = df['code'].str.contains(r'^\s*\d+\.', flags=re.M).astype(int)
    return df

# === 1. Предобработка ===
sample_submission = pd.read_csv('sample_submission.csv')
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['code'] = train['code'].fillna('').astype(str).str.strip()
train = train[train['code'].str.len() > 0]
train = train.dropna(subset=['difficulty'])
train['difficulty'] = train['difficulty'].map({'easy': 0, 'medium': 1, 'hard': 2})

# Сохраним копию train до oversampling для оценки
train_orig = train.copy()

# === 2. Балансировка классов через oversampling ===
max_count = train['difficulty'].value_counts().max()
train_bal = pd.concat([
    train[train['difficulty'] == 0].sample(max_count, replace=True, random_state=42),
    train[train['difficulty'] == 1].sample(max_count, replace=True, random_state=42),
    train[train['difficulty'] == 2].sample(max_count, replace=True, random_state=42)
])

# === 3. Извлечение признаков ===
train_bal = extract_code_features(train_bal)
feature_cols = [
    'char_len', 'word_count', 'sentence_count', 'avg_token_len', 'num_digits',
    'num_loops', 'num_if', 'num_functions', 'num_comments',
    'has_O_complexity', 'contains_tree', 'contains_dp',
    'contains_hash', 'contains_stack', 'contains_recursive', 'has_algo_steps'
]
X_meta = train_bal[feature_cols].values

# === 4. TF-IDF ===
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b"
)
X_tfidf = tfidf.fit_transform(train_bal['code'])

# === 5. Объединяем ===
X_full = hstack([X_tfidf, csr_matrix(X_meta)])
y = train_bal['difficulty']

# === 6. Train / Validation + обучение ===
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y, test_size=0.1, random_state=42, stratify=y
)
model = LGBMClassifier(random_state=42)
model.fit(X_train, y_train)

# === 7. Оценка ===
y_pred = model.predict(X_val)
print('\n[Validation Classification Report on Oversampled Data]')
print(classification_report(y_val, y_pred, target_names=['easy', 'medium', 'hard']))

# === 8. Оценка на оригинальном train (до oversampling) ===
train_orig = extract_code_features(train_orig)
X_orig_meta = train_orig[feature_cols].values
X_orig_tfidf = tfidf.transform(train_orig['code'])
X_orig_full = hstack([X_orig_tfidf, csr_matrix(X_orig_meta)])
y_orig = train_orig['difficulty']
y_orig_pred = model.predict(X_orig_full)

print('\n[Accuracy on Original Train (No Oversampling)]:', accuracy_score(y_orig, y_orig_pred))
print('[Classification Report on Original Train]')
print(classification_report(y_orig, y_orig_pred, target_names=['easy', 'medium', 'hard']))

# === 9. Предсказание на test ===
test['code'] = test['code'].fillna('').astype(str).str.strip()
test = extract_code_features(test)
X_test_meta = test[feature_cols].values
X_test_tfidf = tfidf.transform(test['code'])
X_test_full = hstack([X_test_tfidf, csr_matrix(X_test_meta)])
test_preds = model.predict(X_test_full)

# === 10. Сабмишен ===
difficulty_map = {0: 'easy', 1: 'medium', 2: 'hard'}
sample_submission['difficulty'] = pd.Series(test_preds).map(difficulty_map)
sample_submission.to_csv('submission.csv', index=False)

# === 11. Просмотр ===
print('\n[Test Prediction Distribution]')
print(sample_submission['difficulty'].value_counts())
print('\n[Sample Submission Preview]')
print(sample_submission.head())


In [ ]:

# === [추가] 로컬 채점용: 동봉 검증셋(val.csv) 예측 → submission_val.csv ===
val_df = pd.read_csv("val.csv")
val_df['code'] = val_df['code'].fillna('').astype(str).str.strip()
val_df = extract_code_features(val_df)
X_val_meta = val_df[feature_cols].values
X_val_full = hstack([tfidf.transform(val_df['code']), csr_matrix(X_val_meta)])
val_df['difficulty'] = pd.Series(model.predict(X_val_full)).map(difficulty_map).values
val_df[['id','difficulty']].to_csv('submission_val.csv', index=False)
print("saved submission_val.csv", val_df.shape)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)